In [ ]:
# ------------------------------------------------------------
# CELL 1 · Install dependencies (RUN ONCE)
# ------------------------------------------------------------

!pip uninstall -y transformers sentence-transformers peft accelerate torchcodec -q
!pip install -q transformers==4.41.2
!pip install -q sentence-transformers==3.0.1
!pip install -q faiss-cpu openai pandas openpyxl

print("✅ Packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.6 MB/s eta 0:00:00
✅ Packages installed


In [ ]:
# ------------------------------------------------------------
# CELL 2 · Mount Google Drive
# ------------------------------------------------------------

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ------------------------------------------------------------
# CELL 3 · Project setup
# ------------------------------------------------------------

PROJECT_PATH = "/content/drive/MyDrive/WanderRiyadh"

import os
os.makedirs(PROJECT_PATH, exist_ok=True)

print("✅ Folder ready:", PROJECT_PATH)

✅ Folder ready: /content/drive/MyDrive/WanderRiyadh


In [ ]:
# ------------------------------------------------------------
# CELL 4 · Imports & configuration
# ------------------------------------------------------------

import os
import re
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

# ── Model constants ─────────────────────────────────────────
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM = 384
TOP_K = 5

# ── File paths ──────────────────────────────────────────────
FOOD_FILE = "/content/drive/MyDrive/WanderRiyadh/Food and Beverages in Riyadh CityPJ csv.xlsx"
PLACES_FILE = "/content/drive/MyDrive/WanderRiyadh/Riyadh ExperiencesPJ csv.xlsx"

print("✅ Imports and configuration loaded")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


✅ Imports and configuration loaded


In [ ]:
# ------------------------------------------------------------
# Check dataset loading and columns
# ------------------------------------------------------------

df_food_raw = pd.read_excel(FOOD_FILE)
df_places_raw = pd.read_excel(PLACES_FILE)

print("Food dataset shape:", df_food_raw.shape)
print("Places dataset shape:", df_places_raw.shape)

print("\nFood columns:")
print(df_food_raw.columns.tolist())

print("\nPlaces columns:")
print(df_places_raw.columns.tolist())

display(df_food_raw.head())
display(df_places_raw.head())

Food dataset shape: (129, 10)
Places dataset shape: (99, 8)

Food columns:
['DESTINATION', 'NAME', 'DESCRIPTION', 'LOCATION (Google Maps Link)', 'SM/REFERENCE LINKS', 'Type', 'tags', 'Category', 'price_level', 'Area']

Places columns:
['DESTINATION', 'NAME', 'DESCRIPTION', 'SM/REFERENCE LINKS', 'LOCATION (Google Maps Link)', 'Type', 'Tags', 'Area']


,DESTINATION,NAME,DESCRIPTION,LOCATION (Google Maps Link),SM/REFERENCE LINKS,Type,tags,Category,price_level,Area
0,Riyadh,Aseeb Najd,Aseeb Najd is a restaurant offering authentic ...,https://maps.app.goo.gl/TVyhLZ2TiBmWM5Cd7,https://www.instagram.com/aseeb.najd,Restaurant,"najdi, traditional, local-flavors, saudi-cuisi...",Saudi,Mid-range,"Alyasmin, Riyadh"
1,Riyadh,Somewhere,Somewhere is a contemporary restaurant offerin...,https://maps.app.goo.gl/DfSyu5QE3zbWMati7,https://www.instagram.com/somewhere?igsh=MTJ2d...,Restaurant and Cafe,"fusion, international, modern-dining, trendy, ...",Aragic,Mid-range,"Al Bujairi, Diriyah"
2,Riyadh,AL MAMLAKA Social Dining,AL MAMLAKA Social Dining is a premium restaura...,https://maps.app.goo.gl/o692nSRv7revvG1H9,https://www.instagram.com/almamlakasocialdinin...,Restaurant and Cafe,"international, luxury, premium, crowd","Italian,Lebanese,Japanese,Burgers,Steaks",Expensive,"Kingdom Centre, Olaya St, Al Olaya, Riyadh"
3,Riyadh,Takya,Takya fuses Saudi cuisine with international i...,https://maps.app.goo.gl/ZavDH2JputLKUcbv8,https://www.instagram.com/takya_sa?igsh=bml3cX...,Restaurant,"modern-saudi, family, groups, mathlutha, bali...",Saudi,Mid-range,"King Faisal Rd, Al Bujairi, Riyadh"
4,Riyadh,Maiz,Maiz offers authentic Saudi cuisine with a fin...,https://maps.app.goo.gl/avQuoSsZtoRehE6U7,https://www.instagram.com/maizriyadh/,Restaurant,"kubeba, kabsa, traditional, al-bujairi, upscal...",Saudi,Expensive,"Al Bujairi, Diriyah"


,DESTINATION,NAME,DESCRIPTION,SM/REFERENCE LINKS,LOCATION (Google Maps Link),Type,Tags,Area
0,Riyadh,Riyadh National Zoo,"Visit Riyadh Zoo, Saudi Arabia’s largest and o...",https://www.instagram.com/riyadh_zoo/?hl=en,https://maps.app.goo.gl/34sCFXjAAZe3ydKq7,tourist place,"family, animals, outdoor, kids, park","Salah Al Din Al Ayyubi St, Al Malaz, Riyadh"
1,Riyadh,Via Riyadh,VIA Riyadh offers luxury with the St. Regis ho...,https://www.instagram.com/viariyadh?igshid=MWI...,https://maps.app.goo.gl/F8WZScoeq82hSVRx7,tourist place,"luxury, restaurants, cafes, shopping, entertai...","Al Hada, Riyadh"
2,Riyadh,Murabba Palace,Al Murabba Historical Palace in Riyadh is a sy...,https://www.visitsaudi.com/en/riyadh/attractio...,https://maps.app.goo.gl/puMFxDmYi7XLU8Hf9,tourist place,"heritage, history, palace, culture","King Saud Rd, Al Murabba, Riyadh"
3,Riyadh,Addoho Neighborhood,Addoho Neighborhood is the last old neighborho...,https://www.visitsaudi.com/en/riyadh/attractio...,https://maps.app.goo.gl/W127KHWHzhngPydeA,tourist place,"heritage, traditional, culture","Al Dirah, Riyadh"
4,Riyadh,Wadi Namar Waterfall,Wadi Namar Waterfall and Dam Park in Riyadh is...,https://welcomesaudi.com/activity/wadi-namer-w...,https://maps.app.goo.gl/3gmocWFrwSRvEpsE7,tourist place,"nature, waterfall, lake, picnic, outdoor","Al-Tirmidhi St, Namar, Riyadh"


In [ ]:
# ------------------------------------------------------------
# CELL 5 · Preprocess Food Dataset
# ------------------------------------------------------------

df_food = df_food_raw.copy()

df_food.columns = df_food.columns.str.strip().str.lower()
df_food.drop_duplicates(inplace=True)

food_text_cols = [
    "name",
    "description",
    "tags",
    "type",
    "category",
    "price_level",
    "area"
]

for col in food_text_cols:
    if col in df_food.columns:
        df_food[col] = df_food[col].fillna("")

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return text.strip()

for col in food_text_cols:
    if col in df_food.columns:
        df_food[col] = df_food[col].apply(clean_text)

df_food["dataset_type"] = "food_beverage"

df_food["search_text"] = (
    df_food["name"] + " " +
    df_food["description"] + " " +
    df_food["tags"] + " " +
    df_food["type"] + " " +
    df_food["category"] + " " +
    df_food["price_level"] + " " +
    df_food["area"]
)

print("✅ Food preprocessing done")
print("Food shape:", df_food.shape)

display(df_food[["name", "type", "category", "price_level", "area", "search_text"]].head())

✅ Food preprocessing done
Food shape: (129, 12)


,name,type,category,price_level,area,search_text
0,aseeb najd,restaurant,saudi,mid range,alyasmin riyadh,aseeb najd aseeb najd is a restaurant offering...
1,somewhere,restaurant and cafe,aragic,mid range,al bujairi diriyah,somewhere somewhere is a contemporary restaura...
2,al mamlaka social dining,restaurant and cafe,italian lebanese japanese burgers steaks,expensive,kingdom centre olaya st al olaya riyadh,al mamlaka social dining al mamlaka social din...
3,takya,restaurant,saudi,mid range,king faisal rd al bujairi riyadh,takya takya fuses saudi cuisine with internati...
4,maiz,restaurant,saudi,expensive,al bujairi diriyah,maiz maiz offers authentic saudi cuisine with ...


In [ ]:
# ------------------------------------------------------------
# CELL 6 · Preprocess Places Dataset
# ------------------------------------------------------------

df_places = df_places_raw.copy()

df_places.columns = df_places.columns.str.strip().str.lower()
df_places.drop_duplicates(inplace=True)

places_text_cols = [
    "name",
    "description",
    "tags",
    "type",
    "area"
]

for col in places_text_cols:
    if col in df_places.columns:
        df_places[col] = df_places[col].fillna("")

for col in places_text_cols:
    if col in df_places.columns:
        df_places[col] = df_places[col].apply(clean_text)

df_places["dataset_type"] = "tourism_place"

df_places["search_text"] = (
    df_places["name"] + " " +
    df_places["description"] + " " +
    df_places["tags"] + " " +
    df_places["type"] + " " +
    df_places["area"]
)

print("✅ Places preprocessing done")
print("Places shape:", df_places.shape)

display(df_places[["name", "type", "area", "search_text"]].head())

✅ Places preprocessing done
Places shape: (99, 10)


,name,type,area,search_text
0,riyadh national zoo,tourist place,salah al din al ayyubi st al malaz riyadh,riyadh national zoo visit riyadh zoo saudi ar...
1,via riyadh,tourist place,al hada riyadh,via riyadh via riyadh offers luxury with the s...
2,murabba palace,tourist place,king saud rd al murabba riyadh,murabba palace al murabba historical palace in...
3,addoho neighborhood,tourist place,al dirah riyadh,addoho neighborhood addoho neighborhood is the...
4,wadi namar waterfall,tourist place,al tirmidhi st namar riyadh,wadi namar waterfall wadi namar waterfall and ...


In [ ]:
# ------------------------------------------------------------
# CELL 7 · Combine processed datasets
# ------------------------------------------------------------

df_kb = pd.concat([df_food, df_places], ignore_index=True)

print("✅ Knowledge base created")
print("Final KB shape:", df_kb.shape)

display(df_kb[["name", "dataset_type", "search_text"]].head())

✅ Knowledge base created
Final KB shape: (228, 12)


,name,dataset_type,search_text
0,aseeb najd,food_beverage,aseeb najd aseeb najd is a restaurant offering...
1,somewhere,food_beverage,somewhere somewhere is a contemporary restaura...
2,al mamlaka social dining,food_beverage,al mamlaka social dining al mamlaka social din...
3,takya,food_beverage,takya takya fuses saudi cuisine with internati...
4,maiz,food_beverage,maiz maiz offers authentic saudi cuisine with ...


In [ ]:
# ------------------------------------------------------------
# CELL 8 · Validate and save cleaned knowledge base
# ------------------------------------------------------------

# 1. Check final size
print("Final KB shape:", df_kb.shape)

# 2. Check missing values in important columns
important_cols = ["name", "description", "type", "tags", "area", "search_text", "dataset_type"]

print("\nMissing values in important columns:")
print(df_kb[important_cols].isnull().sum())

# 3. Check empty search_text
empty_search = df_kb[df_kb["search_text"].str.strip() == ""]
print("\nEmpty search_text rows:", len(empty_search))

# 4. Check duplicate names
duplicate_names = df_kb[df_kb.duplicated(subset=["name"], keep=False)]
print("Duplicate place names:", len(duplicate_names))

# 5. Save cleaned knowledge base
CLEANED_KB_FILE = PROJECT_PATH + "/cleaned_knowledge_base.xlsx"
df_kb.to_excel(CLEANED_KB_FILE, index=False)

print("\n✅ Cleaned knowledge base saved to:")
print(CLEANED_KB_FILE)

# 6. Preview
display(df_kb[["name", "dataset_type", "type", "area", "search_text"]].head())

Final KB shape: (228, 12)

Missing values in important columns:
name            0
description     0
type            0
tags            0
area            0
search_text     0
dataset_type    0
dtype: int64

Empty search_text rows: 0
Duplicate place names: 2

✅ Cleaned knowledge base saved to:
/content/drive/MyDrive/WanderRiyadh/cleaned_knowledge_base.xlsx


,name,dataset_type,type,area,search_text
0,aseeb najd,food_beverage,restaurant,alyasmin riyadh,aseeb najd aseeb najd is a restaurant offering...
1,somewhere,food_beverage,restaurant and cafe,al bujairi diriyah,somewhere somewhere is a contemporary restaura...
2,al mamlaka social dining,food_beverage,restaurant and cafe,kingdom centre olaya st al olaya riyadh,al mamlaka social dining al mamlaka social din...
3,takya,food_beverage,restaurant,king faisal rd al bujairi riyadh,takya takya fuses saudi cuisine with internati...
4,maiz,food_beverage,restaurant,al bujairi diriyah,maiz maiz offers authentic saudi cuisine with ...


In [ ]:
# ------------------------------------------------------------
# CELL 9 · Load embedding model
# ------------------------------------------------------------

embed_model = SentenceTransformer(EMBED_MODEL_NAME)

print("✅ Embedding model loaded successfully")
print("Model:", EMBED_MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded successfully
Model: sentence-transformers/all-MiniLM-L6-v2


In [ ]:
# ------------------------------------------------------------
# CELL 10 · Prepare texts for embedding
# ------------------------------------------------------------

texts = df_kb["search_text"].astype(str).tolist()

print("✅ Texts prepared")
print("Number of texts:", len(texts))
print("Example text:")
print(texts[0])

✅ Texts prepared
Number of texts: 228
Example text:
aseeb najd aseeb najd is a restaurant offering authentic najdi cuisine  showcasing traditional flavors from the heart of saudi arabia najdi  traditional  local flavors  saudi cuisine  heritage  kabsa  jareesh  family friendly  quiet restaurant saudi mid range alyasmin  riyadh


In [ ]:
# ------------------------------------------------------------
# CELL 11 · Generate embeddings
# ------------------------------------------------------------

embeddings = embed_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("✅ Embeddings generated")
print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Embeddings generated
Embeddings shape: (228, 384)


In [ ]:
# ------------------------------------------------------------
# CELL 12 · Validate embeddings
# ------------------------------------------------------------

assert embeddings.shape[0] == len(df_kb), "❌ Embeddings count does not match dataset rows"
assert embeddings.shape[1] == EMBED_DIM, "❌ Embedding dimension is not 384"

print("✅ Embeddings validation passed")
print("Rows:", embeddings.shape[0])
print("Dimension:", embeddings.shape[1])

✅ Embeddings validation passed
Rows: 228
Dimension: 384


In [ ]:
# ------------------------------------------------------------
# CELL 13 · Build FAISS index
# ------------------------------------------------------------

index = faiss.IndexFlatIP(EMBED_DIM)  # cosine similarity (لأننا normalized)

index.add(embeddings)

print("✅ FAISS index created")
print("Number of vectors in index:", index.ntotal)

✅ FAISS index created
Number of vectors in index: 228


In [ ]:
# ------------------------------------------------------------
# CELL 14 · Save index and embeddings
# ------------------------------------------------------------

faiss.write_index(index, PROJECT_PATH + "/faiss_index.index")
np.save(PROJECT_PATH + "/embeddings.npy", embeddings)

print("✅ FAISS index and embeddings saved")

✅ FAISS index and embeddings saved


In [ ]:
# ------------------------------------------------------------
# CELL 15 · Test semantic search
# ------------------------------------------------------------

query = "family friendly places in riyadh"

query_embedding = embed_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

scores, indices = index.search(query_embedding, TOP_K)

print("Query:", query)
print("\nTop results:")

for i, idx in enumerate(indices[0]):
    print(f"\nResult {i+1}:")
    print("Name:", df_kb.iloc[idx]["name"])
    print("Type:", df_kb.iloc[idx]["type"])
    print("Area:", df_kb.iloc[idx]["area"])

Query: family friendly places in riyadh

Top results:

Result 1:
Name: big fun family
Type: entertainment
Area: qurtubah  riyadh

Result 2:
Name: riyadh golf club
Type: sport
Area: king abdulaziz rd  al aarid  riyadh

Result 3:
Name: kitchen on 3
Type: restaurant
Area: district  king fahad road  sahafah  riyadh

Result 4:
Name: riyadh national zoo
Type: tourist place
Area: salah al din al ayyubi st  al malaz  riyadh

Result 5:
Name: salam park
Type: park
Area: central riyadh


In [ ]:
# ------------------------------------------------------------
# CELL 16 · OpenAI API setup
# ------------------------------------------------------------

from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get('open_AI_key'))
print("✅ OpenAI client ready")

✅ OpenAI client ready


In [ ]:
# ----------------------------------------------------------
# CELL 17 · Chatbot function
# ----------------------------------------------------------

def build_context(indices, scores):
    context = ""
    for i, idx in enumerate(indices[0]):
        row = df_kb.iloc[idx]
        context += f"""
Place {i+1}:
- Name: {row['name']}
- Type: {row['type']}
- Description: {row['description']}
- Area: {row['area']}
- Tags: {row['tags']}
- Score: {round(float(scores[0][i]), 4)}
"""
    return context


def chat(user_query):
    # 1. Embed the query
    query_embedding = embed_model.encode(
        [user_query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # 2. Retrieve top-k from FAISS
    scores, indices = index.search(query_embedding, TOP_K)

    # 3. Build context
    context = build_context(indices, scores)

    # 4. Generate response with GPT-4o-mini
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are WanderRiyadh, a friendly tourism assistant "
                    "that helps users discover places and experiences in Riyadh, Saudi Arabia. "
                    "Use the provided context to recommend suitable places. "
                    "Be concise, helpful, and friendly."
                )
            },
            {
                "role": "user",
                "content": f"User question: {user_query}\n\nRelevant places:\n{context}"
            }
        ],
        max_tokens=500,
        temperature=0.7
    )

    answer = response.choices[0].message.content
    return answer, indices, scores


print("✅ Chatbot function ready")

✅ Chatbot function ready


In [ ]:
# ----------------------------------------------------------
# CELL 18 · Test the chatbot
# ----------------------------------------------------------

test_queries = [
    "What are some family friendly places in Riyadh?",
    "Recommend a good Saudi restaurant in Riyadh",
    "Where can I find cultural and heritage sites in Riyadh?"
]

for query in test_queries:
    print("=" * 60)
    print(f"Query: {query}")
    print("-" * 60)
    answer, indices, scores = chat(query)
    print(f"Response:\n{answer}")
    print()

print("✅ Chatbot testing done")

Query: What are some family friendly places in Riyadh?
------------------------------------------------------------
Response:
Here are some great family-friendly places in Riyadh:

1. **Big Fun Family** - This indoor entertainment center in Qurtubah offers a wide range of games, rides, and activities designed for children and families. It's perfect for a fun day out!

2. **Kitchen on 3** - Located on King Fahad Road, this restaurant serves a mix of Middle Eastern dishes and global fare in a relaxed environment, making it an excellent choice for family dining.

3. **Meez the House** - A cozy modern Middle Eastern eatery in Hittin, offering homegrown ingredients and a casual atmosphere that's perfect for families.

4. **Riyadh Golf Club** - For families that enjoy outdoor activities, this scenic golf destination provides a peaceful environment to relax and play golf together.

These spots should ensure a fun and enjoyable time for the whole family!

Query: Recommend a good Saudi restaura

In [ ]:
# ----------------------------------------------------------
# CELL 19 · Evaluation (Precision@k and Hit@k)
# ----------------------------------------------------------

# Test queries with their expected relevant keywords
test_cases = [
    {
        "query": "family friendly places in riyadh",
        "relevant_keywords": ["family", "kids", "children", "entertainment", "park"]
    },
    {
        "query": "saudi traditional restaurant",
        "relevant_keywords": ["saudi", "traditional", "najdi", "local", "arabic"]
    },
    {
        "query": "heritage and cultural sites",
        "relevant_keywords": ["heritage", "culture", "history", "museum", "palace"]
    },
    {
        "query": "outdoor activities in riyadh",
        "relevant_keywords": ["outdoor", "nature", "park", "sport", "garden"]
    },
    {
        "query": "coffee shop and cafe in riyadh",
        "relevant_keywords": ["cafe", "coffee", "specialty", "tea", "drinks"]
    }
]

def is_relevant(row, keywords):
    text = str(row["search_text"]).lower()
    return any(kw in text for kw in keywords)

precision_scores = []
hit_scores = []

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)

for case in test_cases:
    query = case["query"]
    keywords = case["relevant_keywords"]

    # Embed query
    query_embedding = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Retrieve top-k
    scores, indices = index.search(query_embedding, TOP_K)

    # Check relevance
    relevant_count = 0
    hit = 0

    for idx in indices[0]:
        row = df_kb.iloc[idx]
        if is_relevant(row, keywords):
            relevant_count += 1
            hit = 1

    precision = relevant_count / TOP_K
    precision_scores.append(precision)
    hit_scores.append(hit)

    print(f"\nQuery: {query}")
    print(f"Relevant retrieved: {relevant_count}/{TOP_K}")
    print(f"Precision@{TOP_K}: {precision:.2f}")
    print(f"Hit@{TOP_K}: {hit}")

print("\n" + "=" * 60)
print(f"Average Precision@{TOP_K}: {sum(precision_scores)/len(precision_scores):.2f}")
print(f"Average Hit@{TOP_K}: {sum(hit_scores)/len(hit_scores):.2f}")
print("=" * 60)
print("✅ Evaluation done")

EVALUATION RESULTS

Query: family friendly places in riyadh
Relevant retrieved: 4/5
Precision@5: 0.80
Hit@5: 1

Query: saudi traditional restaurant
Relevant retrieved: 4/5
Precision@5: 0.80
Hit@5: 1

Query: heritage and cultural sites
Relevant retrieved: 5/5
Precision@5: 1.00
Hit@5: 1

Query: outdoor activities in riyadh
Relevant retrieved: 5/5
Precision@5: 1.00
Hit@5: 1

Query: coffee shop and cafe in riyadh
Relevant retrieved: 5/5
Precision@5: 1.00
Hit@5: 1

Average Precision@5: 0.92
Average Hit@5: 1.00
✅ Evaluation done


In [ ]:
# ----------------------------------------------------------
# CELL 20 · Interactive chat session
# Type 'quit' or 'exit' to stop.
# ----------------------------------------------------------

print("🌟 Welcome to WanderRiyadh Chatbot!")
print("Ask me anything about places and experiences in Riyadh.")
print("Type 'quit' or 'exit' to stop.\n")

while True:
    user_input = input("You: ").strip()

    if not user_input:
        continue

    if user_input.lower() in ["quit", "exit"]:
        print("👋 Thank you for using WanderRiyadh! Goodbye!")
        break

    answer, indices, scores = chat(user_input)
    print(f"\n🤖 WanderRiyadh: {answer}\n")
    print("-" * 60)

🌟 Welcome to WanderRiyadh Chatbot!
Ask me anything about places and experiences in Riyadh.
Type 'quit' or 'exit' to stop.

You: What are the best places to visit in Riyadh?

🤖 WanderRiyadh: Riyadh has some fantastic places to explore! Here are a few recommendations:

1. **Riyadh Golf Club** - If you enjoy golf, this scenic destination offers a relaxing outdoor environment perfect for playing a round or just unwinding in nature. Located on King Abdulaziz Rd, it's a great spot for sports enthusiasts.

2. **Riyadh Park Mall** - For shopping lovers, this is one of the largest malls in Riyadh, featuring a variety of international brands, restaurants, cafes, and entertainment options like cinemas. It's situated on Northern Ring Branch Rd.

3. **Via Riyadh** - Experience luxury at its finest with designer stores, top-notch restaurants, and the exclusive Renaissance cinema. It’s a perfect place for those looking to indulge in a high-end experience, located in Al Hada.

4. **Cool Arena** - Beat